# Day 11 — held-out test run of the frozen agent (100 questions)

The agent has been frozen since Day 7 v2 (anchor_k=5, max_evidence=7, gpt-oss-120b, current prompts), and the
reference-quality filter was fixed on Day 9, so everything is decided **before** we look at test data.
Today we run it once on **100 random test questions** (seed 0) and judge the answers with gpt-oss-20b.
These are the final numbers for the CV ("citation accuracy >85%, evidence recall >75% across 100 research queries").

Rules for a fair test run:
* No code, prompt or setting changes after this run starts, whatever the numbers look like.
* The run takes about 2 days, because the free Groq tier allows ~200k tokens/day for gpt-oss-120b (~65 questions).
  It stops cleanly at the limit; running the same cell again later continues where it stopped (`--resume`).

Use a **GPU** runtime (the reranker runs for every question). The Gemini judge keeps running in its own notebook.

In [ ]:
# WHAT: Mounts Google Drive and makes the research-engine folder the working directory.
# WHY:  Every command below runs from the project root (data/, cache/, results/ and the code live there).
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/research-engine


In [ ]:
# WHAT: Installs the project's packages.
# WHY:  A fresh Colab session doesn't have them. Run this again after every runtime restart.
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 15.7 MB/s eta 0:00:00


In [ ]:
# WHAT: Copies the Groq key from Colab Secrets into an environment variable.
# WHY:  The `!python` commands below run as separate programs and can only see environment variables.
#       (In a new notebook, switch on "Notebook access" for GROQ_API_KEY in the 🔑 panel first.)
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
print("key loaded:", len(os.environ["GROQ_API_KEY"]) > 0)

key loaded: True


In [ ]:
# WHAT: Shows the code version and the agent/retrieval settings, and checks that agent/, retrieval/ and eval/
#       have no local edits.
# WHY:  Proof that the test run uses exactly the frozen code that is on GitHub. The git status line must print
#       nothing; the settings must show anchor_k=5, max_evidence=7, llm_model='openai/gpt-oss-120b'.
!git log --oneline -1
!git status --short agent/ retrieval/ eval/
from agent.config import AgentConfig
from retrieval.config import RetrievalConfig
print(AgentConfig())
print(RetrievalConfig())

e4e452e (HEAD -> main) Day 10: FAISS index for corpus-wide dense search; benchmark vs numpy (flat exact, HNSW speed/recall grid)
Refresh index: 100% (75/75), done.
?? eval/cross_paper_questions.jsonl
?? eval/run_cross_paper_eval.py
?? retrieval/cross_paper.py
AgentConfig(llm_model='openai/gpt-oss-120b', temperature=0.0, reasoning_effort='low', max_sub_questions=3, anchor_k=5, k_per_query=4, max_evidence=7, decompose_max_tokens=1024, max_answer_tokens=2048)
RetrievalConfig(dense_model='BAAI/bge-small-en-v1.5', reranker_model='BAAI/bge-reranker-v2-m3', pool=50, rrf_k=60, rerank_top=30, reranker_max_length=512, reranker_batch_size=64, fp16=True, exclude_sections=('abstract',), corpus_index='flat', hnsw_m=32, hnsw_ef_construction=200, hnsw_ef_search=128)


## Step 1 — the agent on 100 test questions
Output: `results/agent_eval_test_n100_v2.jsonl` (one line per question, written as it goes) and a summary `.json`.
Each printed line shows the strict citation precision and the evidence recall for that question.
When the daily limit is reached it prints **STOPPED … daily quota reached** and saves a partial summary.
That is expected: run this same cell again tomorrow (after the 24-hour window frees tokens).

In [ ]:
# WHAT: Runs the frozen agent on 100 random test questions (seed 0), continuing from saved rows if any.
# WHY:  This is the held-out measurement. --resume makes it safe to run this cell again and again: it never
#       redoes or overwrites finished questions. About 18 s and ~3k tokens per question.
!python -m eval.run_agent_eval --split test --n 100 --tag v2 --resume

modules.json: 100% 349/349 [00:00<00:00, 1.14MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 611kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 118MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 267kB/s]
config.json: 100% 743/743 [00:00<00:00, 4.74MB/s]

model.safetensors: downloading bytes:  53% 70.3M/133M [00:00<00:00, 101MB/s, 5.45MB/s  ] 
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 57.0MB/s, 7.98MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 87.7MB/s, 12.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 7969.70it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.82MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 30.7MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 67.8MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 567kB/s]
config.json: 100% 190/190 [00:00<00:00, 946kB/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.np

In [ ]:
# WHAT: Counts how many of the 100 questions are done.
# WHY:  A quick progress check, without re-running anything.
!wc -l results/agent_eval_test_n100_v2.jsonl

100 results/agent_eval_test_n100_v2.jsonl


## Step 2 — the judge (gpt-oss-20b)
The judge reads the saved answers only (it never re-runs the agent) and has its own daily quota, so it can run
on the questions finished so far and continue tomorrow with `--resume`.

In [ ]:
# WHAT: Judges every saved answer that hasn't been judged yet: does each cited passage support its claim,
#       is the answer faithful to its passages, does it agree with the QASPER reference?
# WHY:  The strict citation metric undercounts (Day 7 review), so the judged citation precision is the CV
#       "citation accuracy" number; strict precision is reported next to it. Output: *_judge-gpt-oss-20b.json(l).
!python -m eval.judge_agent_eval results/agent_eval_test_n100_v2.jsonl --resume

100 answers in agent_eval_test_n100_v2.jsonl | judge=openai/gpt-oss-20b | resuming: 89 already judged
[ 90/100] strict=1.00 judged=1.00 faithful=yes correct=yes  what do they mean by description length?
[ 91/100] strict=0.00 judged=0.75 faithful=yes correct=no  what models did they compare with?
[ 92/100] strict=1.00 judged=1.00 faithful=yes correct=yes  Which morphological features are extracted?
[ 93/100] strict=1.00 judged=0.50 faithful=yes correct=yes  Were the Wikipedia edits that removed framings, presupp
[ 94/100] strict=1.00 judged=1.00 faithful=yes correct=yes  What type of frequency analysis was used?
[ 95/100] strict=0.00 judged=1.00 faithful=yes correct=yes  Do they compare against popular topic models, such as L
[ 96/100] strict=1.00 judged=1.00 faithful=yes correct=yes  by how much is accuracy improved?
[ 97/100] strict=0.00 judged=1.00 faithful=yes correct=yes  What kind of inference model do they build to estimate 
[ 98/100] strict=1.00 judged=1.00 faithful=yes correct=

In [ ]:
# WHAT: Prints the agent and judge summaries side by side: all questions, and the reference-filtered set.
# WHY:  These are the numbers for the results log and the CV. "complete": false means the run is still partial.
import json
for f in ["results/agent_eval_test_n100_v2.json", "results/agent_eval_test_n100_v2_judge-gpt-oss-20b.json"]:
    if os.path.exists(f):
        d = json.load(open(f))
        print(f"\n== {f}  (n={d.get('n')}, complete={d.get('complete', '-')})")
        for k, v in d["summary"].items():
            print(f"  {k:<36} {v}   | filtered: {d.get('summary_filtered', {}).get(k)}")


== results/agent_eval_test_n100_v2.json  (n=100, complete=True)
  citation_precision                   0.7259   | filtered: 0.7259
  citation_hit                         0.8   | filtered: 0.8
  citation_recall                      0.6167   | filtered: 0.6167
  evidence_recall                      0.848   | filtered: 0.848
  single_query_recall                  0.8463   | filtered: 0.8463
  no_citation_rate                     0.03   | filtered: 0.03
  declined_rate                        0.03   | filtered: 0.03
  invalid_citation_rate                0.0   | filtered: 0.0
  avg_sub_questions                    1.24   | filtered: 1.24
  avg_prompt_tokens                    2826   | filtered: 2826
  avg_completion_tokens                181   | filtered: 181
  avg_secs                             64.23   | filtered: 64.23

== results/agent_eval_test_n100_v2_judge-gpt-oss-20b.json  (n=100, complete=True)
  strict_citation_precision            0.7259   | filtered: 0.7259
  judged_citation_p